# CreditNLP Live Demo

**Fine-tuned Mistral-7B for Startup Credit Risk Assessment**

## Setup Instructions
1. **Runtime - Change runtime type - T4 GPU**
2. Upload `creditnlp-lora-adapter.zip` to Colab
3. Run all cells
4. Use the Gradio link for your presentation

In [ ]:
%%capture
!pip install transformers accelerate bitsandbytes peft gradio -q

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
import zipfile
import os

zip_path = "/content/creditnlp-lora-adapter.zip"
if os.path.exists(zip_path):
    print("Found adapter zip file")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall("/content/creditnlp-lora")
    print("Extracted to /content/creditnlp-lora")
else:
    print("Please upload creditnlp-lora-adapter.zip to Colab")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
ADAPTER_PATH = "/content/creditnlp-lora"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

print("Loading base model with 4-bit quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

print("Model loaded successfully!")
print(f"Memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
import re

def predict_signals(application_text: str, temperature: float = 0.3) -> str:
    """Run inference to get risk signal analysis (no binary prediction)."""
    
    prompt = f"""<s>[INST] You are a credit risk analyst evaluating a startup loan application.

Analyze this application and rate EACH of the 5 risk signals. Do NOT make a final prediction - just rate each signal.

Provide your assessment in EXACTLY this format:

TRACTION: [STRONG or MODERATE or WEAK]
Evidence: [Quote the specific phrases that support your rating]

FINANCIAL CLARITY: [STRONG or MODERATE or WEAK]
Evidence: [Quote the specific phrases that support your rating]

BURN RATE: [STRONG or MODERATE or WEAK]
Evidence: [Quote the specific phrases that support your rating]

MANAGEMENT: [STRONG or MODERATE or WEAK]
Evidence: [Quote the specific phrases that support your rating]

MARKET UNDERSTANDING: [STRONG or MODERATE or WEAK]
Evidence: [Quote the specific phrases that support your rating]

Rating Definitions:
- TRACTION: STRONG = signed contracts, named customers, specific revenue. WEAK = "in discussions", "interest", "buzz"
- FINANCIAL CLARITY: STRONG = specific dollar allocations by category. WEAK = "fuel growth", "strategic vision"
- BURN RATE: STRONG = clear runway, path to profitability with milestones. WEAK = reliance on future funding rounds
- MANAGEMENT: STRONG = specific years + titles at named companies in relevant industry. WEAK = "various roles", "diverse backgrounds"
- MARKET UNDERSTANDING: STRONG = specific TAM/SAM with methodology. WEAK = "massive market", "everyone needs this", "infinite TAM"

APPLICATION:
{application_text.strip()}

Rate each signal: [/INST]"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=500,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response


def calculate_risk_level(response: str) -> tuple:
    """Parse the response and calculate overall risk level from signal counts."""
    
    response_upper = response.upper()
    
    signals = ['TRACTION', 'FINANCIAL CLARITY', 'BURN RATE', 'MANAGEMENT', 'MARKET UNDERSTANDING']
    ratings = {'STRONG': 0, 'MODERATE': 0, 'WEAK': 0}
    signal_ratings = {}
    
    for signal in signals:
        pattern = rf'{signal}[:\s]*(STRONG|MODERATE|WEAK)'
        match = re.search(pattern, response_upper)
        if match:
            rating = match.group(1)
            ratings[rating] += 1
            signal_ratings[signal] = rating
    
    weak_count = ratings['WEAK']
    strong_count = ratings['STRONG']
    
    if weak_count >= 4:
        risk_level = "HIGH RISK"
        risk_emoji = "\U0001F534"  # red circle
    elif weak_count >= 2:
        risk_level = "ELEVATED RISK"
        risk_emoji = "\U0001F7E1"  # yellow circle
    elif strong_count >= 4:
        risk_level = "LOW RISK"
        risk_emoji = "\U0001F7E2"  # green circle
    else:
        risk_level = "MODERATE RISK"
        risk_emoji = "\U0001F7E1"  # yellow circle
    
    summary = f"""{risk_emoji} OVERALL: {risk_level}
==============================
STRONG: {ratings['STRONG']} signals
MODERATE: {ratings['MODERATE']} signals
WEAK: {ratings['WEAK']} signals
==============================
Final decision remains with underwriter."""
    
    return summary, signal_ratings


# Quick test
test_app = """
PropertyPulse is a proptech startup. Our team combines diverse backgrounds. 
I spent several years in various roles across different industries.
The commercial real estate market is absolutely massive - $20 trillion globally. 
The total addressable market is essentially infinite.
We're in discussions with potential strategic partners.
The funds will fuel our growth initiatives.
"""

print("Testing signal analysis...")
print("=" * 60)
result = predict_signals(test_app)
print(result)
print("=" * 60)
summary, ratings = calculate_risk_level(result)
print("CALCULATED RISK LEVEL:")
print(summary)

In [ ]:
import gradio as gr

SAMPLES = {
    "High Risk - PropertyPulse (PropTech)": """I am requesting $100,000 for PropertyPulse, a proptech startup that will transform how commercial real estate transactions are conducted.

Our platform leverages blockchain technology and smart contracts to streamline property transfers, lease management, and due diligence processes. We're building the infrastructure layer that will power the next generation of real estate transactions.

Our team combines diverse backgrounds and perspectives. I spent several years in various roles across different industries before discovering my passion for real estate technology. My co-founder has experience in both corporate and startup environments, bringing adaptability to our approach. Our technical lead has worked on multiple projects involving emerging technologies and is excited about blockchain's potential. Together, we bring complementary skills and a shared vision.

The commercial real estate market is absolutely massive - we're talking about a $20 trillion asset class globally. Every property transaction, every lease agreement, every building sale could potentially use our platform. This is a once-in-a-generation opportunity to capture a piece of a market that affects literally everyone who works in an office building or shops in a retail space. The total addressable market is essentially infinite when you consider all the applications.

We've been generating significant buzz in the proptech community. Several prominent real estate firms have expressed strong interest in our vision, and we're in discussions with potential strategic partners who see the value we could bring. Industry thought leaders have been receptive to our pitch, and we've had encouraging conversations with innovation teams at major brokerages. The momentum is building.

The funds will fuel our growth initiatives and help us execute on our strategic vision. We plan to invest in platform development, market expansion, and team building to position ourselves for success. The capital will enable us to scale operations and pursue the partnerships that will define our trajectory. We're building for the long term and expect continued investor interest as we demonstrate progress.""",
    
    "Low Risk - MedTrack Solutions (HealthTech)": """I am applying for a $750,000 loan to accelerate growth at MedTrack Solutions, a healthcare technology company I founded to transform how hospitals manage chronic disease patients.

Our platform provides real-time patient monitoring and predictive alerts for care teams, reducing hospital readmissions by identifying at-risk patients before they deteriorate. We integrate with all major EHR systems and deploy within existing clinical workflows.

Our leadership team brings decades of relevant experience. I spent 14 years at one of the nation's largest hospital networks, most recently as VP of Clinical Informatics where I oversaw technology for 186 facilities and 43,000 beds. My co-founder served as Chief Technology Officer at a leading healthcare IT vendor for 11 years, building the analytics platform now used by over 2,000 hospitals nationwide. Our Chief Medical Officer practiced internal medicine for 22 years and led quality improvement initiatives at a top-10 academic medical center. Our VP of Sales previously built the enterprise team at a healthcare analytics unicorn, growing revenue from $8M to $67M ARR.

Our traction demonstrates strong product-market fit. We currently serve 34 hospitals across 3 health systems, generating $89K in monthly recurring revenue. Our flagship customer, a 12-hospital regional system, signed a 3-year enterprise agreement worth $1.4M. We recently closed a pilot-to-production conversion with a 340-bed academic medical center worth $156K annually, and our pipeline includes signed LOIs from 2 additional health systems representing $580K in potential ARR. Net revenue retention is 118% as existing customers expand to additional facilities.

The loan proceeds will be allocated as follows: $300,000 (40%) for engineering to build our ICU-specific module and expand platform capabilities, $225,000 (30%) for sales team expansion from 4 to 7 representatives, $150,000 (20%) for customer success and implementation resources, and $75,000 (10%) for infrastructure and compliance certifications. At our current burn rate of $58,000 per month, this provides 12.9 months of runway. We project reaching cash-flow break-even at $175K MRR, targeted for Q2 2026, with clear milestones: ICU module launch by month 4, 50 total hospital deployments by month 8, and profitability by month 14.""",

    "Borderline - GreenGrid Systems (CleanTech)": """I am applying for a $400,000 bridge loan for GreenGrid Systems, a cleantech company developing smart energy management solutions for commercial buildings.

Our IoT platform monitors and optimizes HVAC, lighting, and electrical systems in real-time, reducing energy consumption by 15-30% while improving occupant comfort. We install proprietary sensors that communicate with our cloud-based AI to make continuous adjustments based on occupancy patterns, weather forecasts, and utility pricing.

I launched GreenGrid because sustainability has been my lifelong passion. After 12 years as an environmental studies professor, I made the leap to entrepreneurship to have direct impact on carbon emissions rather than just studying them. My co-founder is a self-taught electrical engineer who developed our initial hardware prototypes in a home workshop, learning through trial and error. Our head of sales recently transitioned from a career in nonprofit fundraising, bringing mission-driven communication skills. We're first-time founders united by our commitment to fighting climate change.

We're focused on a specific market segment: Class B office buildings between 50,000-200,000 square feet in metropolitan areas with high electricity costs. According to the 2024 CBRE Commercial Real Estate Report, there are 47,000 buildings matching this profile in the top 25 US markets, with average annual energy spend of $340,000, representing a TAM of $16B. Our initial beachhead is California and the Northeast, where utility rates and sustainability mandates create strong buying motivation, a SAM of $4.2B across 12,400 target properties.

Our technology has generated tremendous interest in the sustainability community. We're in promising discussions with several major property management companies about pilot programs. Multiple building owners have expressed enthusiasm about our approach, and we have warm introductions to facilities directors at prominent REITs. Our advisory board has helped open doors to key decision-makers across the industry.

The bridge financing will support our continued development and go-to-market efforts. We'll invest in engineering, sales, and operations to build momentum heading into our next funding round. This capital positions us well to demonstrate traction that will attract Series A investors. We're confident that continued investor interest will enable us to raise a substantial round within the next 12-18 months as we prove out the model."""
}


def analyze_application(text, sample_choice):
    if sample_choice and sample_choice != "-- Custom Input --":
        text = SAMPLES[sample_choice]
    
    if not text.strip():
        return "Please enter an application or select a sample", "", text
    
    response = predict_signals(text, temperature=0.3)
    summary, ratings = calculate_risk_level(response)
    
    return summary, response, text


with gr.Blocks(title="CreditNLP Demo", theme=gr.themes.Soft()) as demo:
    
    gr.Markdown("""
    # CreditNLP: AI Risk Signal Analysis
    
    **Fine-tuned Mistral-7B for Startup Loan Risk Assessment**
    
    CreditNLP scores loan applications across 5 qualitative risk factors.
    
    | Signal | STRONG | WEAK |
    |--------|--------|------|
    | Traction | Signed contracts, named customers | "In discussions", "interest" |
    | Financial Clarity | Specific $ allocations | "Fuel growth", "strategic vision" |
    | Burn Rate | Path to profitability | Reliance on future funding |
    | Management | Years + titles at named companies | "Various roles", "diverse backgrounds" |
    | Market | Specific TAM/SAM with methodology | "Massive market", "infinite TAM" |
    
    **CreditNLP scores the factors. The final decision stays with the underwriter.**
    """)
    
    with gr.Row():
        with gr.Column(scale=3):
            sample_dropdown = gr.Dropdown(
                choices=["-- Custom Input --"] + list(SAMPLES.keys()),
                value="-- Custom Input --",
                label="Select Sample Application",
            )
            
            text_input = gr.Textbox(
                label="Loan Application Narrative",
                placeholder="Paste startup loan application here...",
                lines=15,
            )
            
            analyze_btn = gr.Button("Analyze Risk Signals", variant="primary", size="lg")
        
        with gr.Column(scale=2):
            verdict_output = gr.Textbox(
                label="Overall Risk Level",
                lines=8,
                interactive=False,
            )
            
            reasoning_output = gr.Textbox(
                label="Signal-by-Signal Analysis",
                lines=18,
                interactive=False,
            )
    
    gr.Markdown("""
    ---
    **Risk Level Calculation:** HIGH = 4+ WEAK | ELEVATED = 2-3 WEAK | LOW = 4+ STRONG
    
    **Demo by Paulo Cavallo** | CreditNLP Project
    """)
    
    def load_sample(choice):
        if choice == "-- Custom Input --":
            return ""
        return SAMPLES.get(choice, "")
    
    sample_dropdown.change(fn=load_sample, inputs=[sample_dropdown], outputs=[text_input])
    analyze_btn.click(fn=analyze_application, inputs=[text_input, sample_dropdown], outputs=[verdict_output, reasoning_output, text_input])

print("Launching demo...")
demo.launch(share=True, debug=False)